# Organizing QuakeML Data: A Step-by-Step Guide

Welcome! In this notebook, we are going to learn how to open a file containing earthquake data (called a **QuakeML** file) and clean it up. 

Earthquakes create vibrations called seismic waves. Seismologists record the exact time these waves arrive at different stations. These recorded times are called **picks**. Sometimes, we have multiple picks for the exact same wave because both computer algorithms and human scientists try to estimate the arrival time.

Our goal is to read these picks, group them by station, and select the *best* one to save into a clean CSV spreadsheet.

### Step 1: Importing Our Tools
Before we start, we need to load three helpful tools:
- `obspy`: A special Python toolkit made for seismologists. It knows exactly how to read earthquake files.
- `csv`: A tool to create and write to Comma-Separated Values files (which open nicely in Excel).
- `defaultdict`: A smarter version of a normal Python dictionary. It automatically creates empty lists if we try to add something to a key that doesn't exist yet.

In [1]:
import obspy
import csv
from collections import defaultdict

### Step 2: Reading the Earthquake File
We use `obspy` to open our XML file. It loads all the earthquakes into a giant list called a **Catalog** (`cat`).

In [2]:
# Define the path to our data file
xml_file = "filtered_events.xml"

# Read the file into a catalog
cat = obspy.read_events(xml_file)

print(f"Number of earthquakes found: {len(cat)}")

Number of earthquakes found: 300


### Step 3: Preparing Our Output Spreadsheet
We want to save our final cleaned data. Here, we create a new file called `filtered_picks_organized.csv` and write the column names (the header) at the very top.

In [3]:
csv_filename = "filtered_picks_organized.csv"
csvfile = open(csv_filename, mode='w', newline='')
writer = csv.writer(csvfile)

# Write the column names as the first row
writer.writerow(['event_id', 'origin_time', 'latitude', 'longitude', 'depth_m', 'station', 'phase', 'pick_time', 'method_id'])

print(f"Spreadsheet {csv_filename} is ready to be filled!")

Spreadsheet filtered_picks_organized.csv is ready to be filled!


### Step 4: Extracting Core Earthquake Information
Now we loop through every single earthquake (event) in our catalog.

For each event, we want to grab its core physical details:
- **Origin Time**: When the earthquake started.
- **Latitude & Longitude**: Where it happened on Earth.
- **Depth**: How deep underground it was.

*(Note: Not all events have this data, so we use `if event.origins` to check if it exists first!)*

Here, we only use 1 event as example.

When running the script for real, use the [02_organize_quakexml_file.py](02_organize_quakexml_file.py) to organize all the events.

In [9]:
# We will only look at the first event to demonstrate!
event = cat[0] 

# Clean up the event ID by removing 'smi:local/'
event_id = str(event.resource_id).replace("smi:local/", "")

# Extract core information if it exists
origin_time = str(event.origins[0].time) if event.origins else ""
lat = str(event.origins[0].latitude) if event.origins else ""
lon = str(event.origins[0].longitude) if event.origins else ""
depth = str(event.origins[0].depth) if event.origins else ""

print(f"Event ID: {event_id}")
print(f"Origin Time: {origin_time}")
print(f"Location: {lat}, {lon}")
print(f"Depth: {depth} meters")

Event ID: be6c4b02-1214-424a-a2f0-27869e5b60d7
Origin Time: 2020-03-09T02:49:17.460000Z
Location: -76.601161, -103.462207
Depth: 1930.0 meters


### Step 5: Grouping the Wave Picks
An earthquake produces different types of waves (like **P-waves** which are fast, and **S-waves** which are slower). 
We need to group our recordings (picks) so that we know exactly which station recorded which wave.

We use a nested dictionary for this. The structure looks like this:
`{'Station_Name': {'P_Wave': [pick1, pick2], 'S_Wave': [pick1]}}`

In [5]:
# Create an empty, smart dictionary
organized_picks = defaultdict(lambda: defaultdict(list))

# If this event has picks, group them!
if event.picks:
    for pick in event.picks:
        station = pick.waveform_id.station_code if pick.waveform_id else "Unknown"
        phase = pick.phase_hint # 'P' or 'S'
        
        # Add the pick into its specific bucket
        organized_picks[station][phase].append(pick)

print("Picks have been successfully grouped by station and wave phase!")

Picks have been successfully grouped by station and wave phase!


### Step 6: Filtering for the Best Pick
Here is the most important logic! If a station has multiple guesses for when the P-wave hit, which one do we trust? 

We set up a priority list:
1. **Modelled**: We trust human-reviewed models the most.
2. **Autopick**: If a human hasn't reviewed it, we trust the computer's guess.
3. **Fallback**: If neither label exists, we just grab the first one we see.

In [6]:
# Go through every station
for station, phases in organized_picks.items():
    
    # Go through the P and S waves for that station
    for phase, picks in phases.items():
        selected_pick = None
        
        if len(picks) > 1:
            # PRIORITY 1: Look for 'modelled'
            for p in picks:
                method = str(p.method_id) if p.method_id else ""
                if "modelled" in method:
                    selected_pick = p
                    break
                    
            # PRIORITY 2: Look for 'autopick'
            if not selected_pick:
                for p in picks:
                    method = str(p.method_id) if p.method_id else ""
                    if "autopick" in method:
                        selected_pick = p
                        break
            
            # PRIORITY 3: Fallback to the first one
            if not selected_pick:
                selected_pick = picks[0]
        else:
            # If there's only 1 pick, our job is easy! Just use it.
            selected_pick = picks[0]
            
        # Clean up the method name
        method = str(selected_pick.method_id).replace("smi:local/", "") if selected_pick.method_id else "Unknown"
        
        print(f"Station {station} recorded a {phase}-wave at {selected_pick.time} (Method: {method})")
        
        # Save this winning pick to our CSV spreadsheet!
        writer.writerow([event_id, origin_time, lat, lon, depth, station, phase, str(selected_pick.time), method])

Station DRSC recorded a P-wave at 2020-03-09T02:49:18.086651Z (Method: modelled)
Station DRSC recorded a S-wave at 2020-03-09T02:49:18.681811Z (Method: modelled)


### Step 7: Wrapping Up
Once we have looped through all events and all their picks, we need to close the file to save it properly.

In [7]:
csvfile.close()
print("All done! Data has been cleaned and saved.")

All done! Data has been cleaned and saved.
